# 10 — Visualization

**Prerequisites:** Run [08_perturbation_effects.ipynb](08_perturbation_effects.ipynb) and [09_mixscape.ipynb](09_mixscape.ipynb) first.
Requires: `data/norman_mixscape.h5ad`, `data/de_results.pkl`, `data/perturbation_scores.csv`.

**What you'll learn:**
- Standard Perturb-seq visualizations: volcano plots, clustermaps, perturbation UMAPs
- How to make publication-quality figures with consistent styling
- Pathway enrichment visualization with `decoupler-py` and `gseapy`
- `pertpy` built-in plotting functions
- UpSet plots for comparing overlapping DEG sets

**Estimated time:** 2.5 hours

In [ ]:
import os
import pickle
import scanpy as sc
import pertpy as pt
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from adjustText import adjust_text
import scipy.sparse
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Publication style
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 120,
})

# Load data
ms_path = os.path.join(DATA_DIR, "norman_mixscape.h5ad")
adata = sc.read_h5ad(ms_path) if os.path.exists(ms_path) else sc.read_h5ad(os.path.join(DATA_DIR, "norman_processed.h5ad"))

de_path = os.path.join(DATA_DIR, "de_results.pkl")
if os.path.exists(de_path):
    with open(de_path, "rb") as f:
        de_results = pickle.load(f)
    print(f"DE results: {de_results.shape}")
else:
    de_results = None
    print("DE results not found — run notebook 08 first")

PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"
print(f"Loaded {adata.shape[0]} cells, {adata.obs[PERT_COL].nunique()} perturbations")

## 1. Volcano plots

In [ ]:
def volcano_plot(de_df, gene_col="gene", lfc_col="log2FoldChange",
                 padj_col="padj", title="", lfc_thresh=0.5, padj_thresh=0.05,
                 n_label=10, ax=None):
    """Publication-quality volcano plot with non-overlapping gene labels."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    
    df = de_df.copy().dropna(subset=[lfc_col, padj_col])
    df["-log10_padj"] = -np.log10(df[padj_col].clip(lower=1e-300))
    
    # Categorize points
    sig_up = (df[padj_col] < padj_thresh) & (df[lfc_col] > lfc_thresh)
    sig_down = (df[padj_col] < padj_thresh) & (df[lfc_col] < -lfc_thresh)
    non_sig = ~(sig_up | sig_down)
    
    ax.scatter(df.loc[non_sig, lfc_col], df.loc[non_sig, "-log10_padj"],
               c="#BBBBBB", alpha=0.5, s=8, linewidths=0, rasterized=True)
    ax.scatter(df.loc[sig_up, lfc_col], df.loc[sig_up, "-log10_padj"],
               c="#E53935", alpha=0.7, s=12, linewidths=0, label=f"Up ({sig_up.sum()})", rasterized=True)
    ax.scatter(df.loc[sig_down, lfc_col], df.loc[sig_down, "-log10_padj"],
               c="#1E88E5", alpha=0.7, s=12, linewidths=0, label=f"Down ({sig_down.sum()})", rasterized=True)
    
    # Threshold lines
    ax.axhline(-np.log10(padj_thresh), color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.axvline(lfc_thresh, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.axvline(-lfc_thresh, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    
    # Labels for top genes
    sig_genes = df[sig_up | sig_down].copy()
    top_genes = sig_genes.nsmallest(n_label, padj_col)
    
    texts = []
    for _, row in top_genes.iterrows():
        texts.append(ax.text(row[lfc_col], row["-log10_padj"],
                             row[gene_col], fontsize=7))
    
    if texts:
        try:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
        except:
            pass  # adjustText may fail in some environments
    
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10(padj)")
    ax.set_title(title)
    ax.legend(framealpha=0.5, loc="upper left")
    
    return ax


if de_results is not None:
    # Plot volcanos for top 4 perturbations
    scores = pd.read_csv(os.path.join(DATA_DIR, "perturbation_scores.csv"), index_col=0) \
        if os.path.exists(os.path.join(DATA_DIR, "perturbation_scores.csv")) else None
    
    if scores is not None:
        top4_perts = scores.nlargest(4, "n_sig_total").index.tolist()
    else:
        top4_perts = de_results["gene_target"].value_counts().head(4).index.tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for ax, pert in zip(axes.flat, top4_perts):
        pert_de = de_results[de_results["gene_target"] == pert]
        gene_col = "gene" if "gene" in pert_de.columns else pert_de.columns[0]
        volcano_plot(pert_de, gene_col=gene_col, title=f"{pert} vs. NTC", ax=ax)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "10_volcano_top4.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping volcano plots — load de_results by running notebook 08")

## 2. Heatmap: top DEGs across perturbations

In [ ]:
if de_results is not None:
    gene_col = "gene" if "gene" in de_results.columns else de_results.columns[0]
    
    # Build LFC matrix: perturbations × genes
    # Select top 50 genes by maximum |LFC| across any perturbation
    lfc_wide = de_results.pivot_table(
        index="gene_target",
        columns=gene_col,
        values="log2FoldChange",
        aggfunc="mean",
    ).fillna(0)
    
    # Select top genes
    max_lfc_per_gene = lfc_wide.abs().max(axis=0)
    top_genes = max_lfc_per_gene.nlargest(50).index
    lfc_sub = lfc_wide[top_genes]
    
    # Clustermap
    g = sns.clustermap(
        lfc_sub,
        cmap="RdBu_r",
        center=0,
        vmin=-3,
        vmax=3,
        figsize=(16, 10),
        xticklabels=True,
        yticklabels=True,
        row_cluster=True,
        col_cluster=True,
        cbar_kws={"label": "log2 Fold Change"},
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=7, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)
    g.fig.suptitle("Top 50 DEGs across perturbations (hierarchically clustered)", y=1.01)
    
    g.savefig(os.path.join(FIG_DIR, "10_clustermap.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    print("Perturbations that cluster together have similar transcriptional signatures.")
    print("Genes that cluster together are co-regulated across perturbations.")

## 3. Perturbation UMAP

Instead of plotting individual cells, we can plot **perturbations** in a 2D space where each point is one perturbation and its position reflects the overall transcriptional signature. This compresses the entire dataset into a perturbation-level view.

Construction:
1. Build a perturbations × top DEGs LFC matrix
2. PCA on this matrix
3. UMAP on the PCA coordinates

In [ ]:
if de_results is not None and lfc_wide is not None:
    from sklearn.preprocessing import StandardScaler
    import umap
    
    # PCA on the perturbation × gene LFC matrix
    X_pert = lfc_wide.values
    X_scaled = StandardScaler().fit_transform(X_pert)
    
    n_components = min(20, X_scaled.shape[0] - 1, X_scaled.shape[1] - 1)
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    # UMAP
    try:
        reducer = umap.UMAP(n_components=2, n_neighbors=5, min_dist=0.3, random_state=42)
        X_umap = reducer.fit_transform(X_pca)
    except ImportError:
        print("umap-learn not available — using PCA components directly")
        X_umap = X_pca[:, :2]
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Color by effect size
    if scores is not None:
        effect_sizes = scores.loc[lfc_wide.index, "n_sig_total"].fillna(0)
        sc_plot = ax.scatter(X_umap[:, 0], X_umap[:, 1],
                             c=effect_sizes, cmap="YlOrRd", s=60, alpha=0.8, edgecolors="gray", linewidths=0.3)
        plt.colorbar(sc_plot, ax=ax, label="Number of significant DEGs")
    else:
        ax.scatter(X_umap[:, 0], X_umap[:, 1], c="steelblue", s=60, alpha=0.7)
    
    # Label perturbations
    texts = []
    for i, pert in enumerate(lfc_wide.index):
        texts.append(ax.text(X_umap[i, 0], X_umap[i, 1], pert, fontsize=7))
    
    try:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
    except:
        pass
    
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title("Perturbation UMAP — each point is one perturbation")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "10_perturbation_umap.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    print("Nearby perturbations in this UMAP have similar transcriptional signatures.")
    print("Clusters may correspond to shared pathways, protein complexes, or genetic interactions.")

## 4. Pathway enrichment visualization

In [ ]:
try:
    import gseapy as gp
    
    if de_results is not None:
        # Run ORA for the top perturbation
        top_pert = deg_summary.index[0] if 'deg_summary' in dir() else de_results["gene_target"].value_counts().index[0]
        top_pert_de = de_results[de_results["gene_target"] == top_pert]
        gene_col = "gene" if "gene" in top_pert_de.columns else top_pert_de.columns[0]
        
        # Get upregulated genes
        up_genes = top_pert_de[
            (top_pert_de["padj"] < 0.05) & (top_pert_de["log2FoldChange"] > 0.5)
        ][gene_col].tolist()
        
        if len(up_genes) >= 5:
            print(f"Running ORA for {top_pert} — {len(up_genes)} upregulated genes")
            enr = gp.enrichr(
                gene_list=up_genes,
                gene_sets="MSigDB_Hallmark_2020",
                organism="human",
                outdir=None,
                verbose=False,
            )
            
            top_pathways = enr.results.sort_values("Adjusted P-value").head(15)
            
            fig, ax = plt.subplots(figsize=(8, 6))
            colors = ["#E53935" if p < 0.05 else "#90A4AE" for p in top_pathways["Adjusted P-value"]]
            ax.barh(range(len(top_pathways)), -np.log10(top_pathways["Adjusted P-value"]),
                    color=colors, edgecolor="none")
            ax.set_yticks(range(len(top_pathways)))
            ax.set_yticklabels(top_pathways["Term"], fontsize=9)
            ax.axvline(-np.log10(0.05), color="gray", linestyle="--", linewidth=1)
            ax.set_xlabel("-log10(Adjusted P-value)")
            ax.set_title(f"Pathway enrichment: {top_pert} upregulated genes")
            plt.tight_layout()
            plt.savefig(os.path.join(FIG_DIR, "10_pathway_enrichment.png"), dpi=150, bbox_inches="tight")
            plt.show()
        else:
            print(f"Too few upregulated genes ({len(up_genes)}) for ORA.")
            
except ImportError:
    print("gseapy not available — skipping pathway enrichment")
    print("Install with: pip install gseapy")
except Exception as e:
    print(f"Pathway enrichment failed: {e}")

## 5. UpSet plot: overlapping DEG sets

In [ ]:
try:
    from upsetplot import UpSet, from_memberships
    
    if de_results is not None:
        gene_col = "gene" if "gene" in de_results.columns else de_results.columns[0]
        
        # Build sets of significant genes per perturbation (top 6)
        top_perts = de_results["gene_target"].value_counts().head(6).index.tolist()
        
        deg_sets = {}
        for pert in top_perts:
            pert_de = de_results[de_results["gene_target"] == pert]
            sig_genes = pert_de[
                (pert_de["padj"] < 0.05) & (pert_de["log2FoldChange"].abs() > 0.5)
            ][gene_col].tolist()
            if sig_genes:
                deg_sets[pert] = set(sig_genes)
        
        if len(deg_sets) >= 2:
            # Convert to memberships format
            all_genes = set.union(*deg_sets.values())
            memberships = []
            for gene in all_genes:
                membership = tuple(pert for pert, genes in deg_sets.items() if gene in genes)
                memberships.append(membership)
            
            upset_data = from_memberships(memberships)
            
            fig = plt.figure(figsize=(12, 6))
            upset = UpSet(upset_data, subset_size="count", show_counts=True, min_subset_size=2)
            upset.plot(fig=fig)
            plt.suptitle("Overlapping DEGs across top perturbations")
            plt.savefig(os.path.join(FIG_DIR, "10_upset_degs.png"), dpi=150, bbox_inches="tight")
            plt.show()
            
            # Find shared DEGs
            if len(deg_sets) >= 2:
                shared = set.intersection(*deg_sets.values())
                print(f"Genes significant across all {len(deg_sets)} perturbations: {len(shared)}")
                if shared:
                    print(sorted(shared)[:20])

except ImportError:
    print("upsetplot not available — install with: pip install upsetplot")
except Exception as e:
    print(f"UpSet plot failed: {e}")

## Key takeaways

1. Volcano plots with `adjustText` for non-overlapping labels are the standard per-perturbation DE visualization
2. Clustermaps of perturbation × DEG LFC matrices reveal co-regulated gene programs and functionally similar perturbations
3. The perturbation UMAP (built from the LFC matrix, not single cells) is a powerful way to explore the entire screen at once — nearby points = similar transcriptional effects
4. Pathway enrichment (ORA or GSEA) provides biological interpretation; `gseapy` gives access to MSigDB gene sets in Python
5. UpSet plots are superior to Venn diagrams for comparing DEG sets across >3 perturbations

**Next:** [11_genetic_interactions.ipynb](11_genetic_interactions.ipynb)